In [ ]:
# =============================================
# IMU ZIP → CSV Load → Move Segments → (Event‑Anchored) Time‑Normalize to L → NPZ
# Embedded‑ready options:
#   • TIME_MODE='stretch'  : split at event, resample left/right to hit anchor & L (mask=all‑ones)
#   • TIME_MODE='padcrop'  : original pad/crop with mask (for backward compatibility)
#   • Gravity align uses LPF+Rodrigues (scipy optional; fallback provided)
# Saves:
#   • move_pad_fs{FS}_L{L}_{TIME_MODE}.npz  (X_train/val/test, y_*, mask_*)
#   • scaler.json (mu/sigma, FS, L, etc.)
#   • deploy_config.json (pipeline flags; MCU에 복제 시 참고)
#   • rep_calib.npz (대표 샘플 일부: 양자화/검증용)
# =============================================

# ---------- 0) Drive mount ----------
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('[WARN] Drive mount skipped or failed:', e)

# ---------- 1) Paths / Imports ----------
from pathlib import Path
import os, re, json, zipfile
import numpy as np
import pandas as pd

ROOT_DIR    = Path('/content/drive/MyDrive/IMU_DATA')
EXTRACT_DIR = Path('/content/IMU_DATA_extracted'); EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 2) Auto‑find newest ZIP & extract ----------
zip_candidates = sorted(ROOT_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_candidates:
    ZIP_PATH = zip_candidates[0]
    print(f"[INFO] 사용할 ZIP: {ZIP_PATH.name}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf: zf.extractall(EXTRACT_DIR)
    print(f"[OK] 압축 해제 완료 → {EXTRACT_DIR.resolve()}")
else:
    print(f"[WARN] ZIP을 찾지 못했습니다: {ROOT_DIR}  (이미 해제된 폴더를 사용합니다)")

# ---------- 3) Detect dataset root (common parent of label_* dirs) ----------

def _is_label_dir(p: Path):
    name = p.name.lower()
    return (name.startswith('label_') or name.startswith('class_') or name in {'good','bad','positive','negative'})


def _list_label_dirs(base: Path):
    return [d for d in base.rglob('*') if d.is_dir() and _is_label_dir(d)]


def _common_parent(paths):
    if not paths: return None
    import os as _os
    return Path(_os.path.commonpath([str(p) for p in paths]))


def find_dataset_root(extracted: Path, fallback_root: Path):
    label_dirs = _list_label_dirs(extracted)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto): {parent}")
        return parent
    label_dirs = _list_label_dirs(fallback_root)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto from DRIVE): {parent}")
        return parent
    raise FileNotFoundError("[ERROR] 'label_*' 또는 유사 폴더를 찾지 못했습니다.")

DATASET_ROOT = find_dataset_root(EXTRACT_DIR, ROOT_DIR)

# ---------- 4) Label mapping + robust CSV load ----------

def _infer_label_map(root: Path):
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir() and _is_label_dir(d)]
    if not subdirs:
        subdirs = sorted([d for d in root.rglob('*') if d.is_dir() and _is_label_dir(d)], key=lambda p: p.as_posix())
    if not subdirs:
        raise FileNotFoundError('[ERROR] 라벨 디렉토리를 찾지 못했습니다.')
    mapping = {}
    for d in subdirs:
        name = d.name
        m = re.match(r'^(?:label|class)_(\d+)$', name, flags=re.IGNORECASE)
        if m: y = int(m.group(1))
        else:
            order_bias = {'bad':0, 'negative':0, 'good':1, 'positive':1}
            y = order_bias.get(name.lower(), None)
        mapping[name] = y
    used = {v for v in mapping.values() if v is not None}
    next_ids = [i for i in range(len(mapping)) if i not in used]
    for k in sorted([k for k,v in mapping.items() if v is None]):
        mapping[k] = next_ids.pop(0)
    return mapping


def _read_csv_safely(path: Path, encoding_pref=('utf-8-sig','cp949','utf-8')):
    last_err = None
    for enc in encoding_pref:
        try: return pd.read_csv(path, encoding=enc)
        except Exception as e: last_err = e
    try: return pd.read_csv(path)
    except Exception: raise last_err


def load_labeled_imu(root_dir: Path, label_map='auto', pattern=('*.csv','*.CSV'), verbose=True, preview_files=5):
    root = Path(root_dir); assert root.exists(), f"[ERROR] 루트가 없습니다: {root}"
    label_map_used = _infer_label_map(root) if label_map=='auto' else dict(label_map)
    print(f"[INFO] 라벨 매핑: {label_map_used}")
    total_files, dfs = 0, []
    for subdir_name, y in label_map_used.items():
        d = root / subdir_name
        if not (d.exists() and d.is_dir()):
            print(f"[MISS] 폴더 없음: {d}"); continue
        files = sorted({f for pat in pattern for f in d.glob(pat)})
        n = len(files); total_files += n
        print(f"[OK] {d.name} → {n}개 파일")
        if verbose and n>0:
            for f in files[:preview_files]: print(f"    • {f.name}")
            if n > preview_files: print(f"    • ... (총 {n}개)")
        for f in files:
            try: df = _read_csv_safely(f)
            except Exception as e: print(f"[WARN] 읽기 실패: {f.name} ({e})"); continue
            df['label']=y; df['source_file']=f.name; df['source_dir']=subdir_name; dfs.append(df)
    if total_files==0 or not dfs:
        raise FileNotFoundError('[ERROR] CSV를 찾지 못했습니다. 경로/패턴을 확인하세요.')
    data = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

    # quick summary
    def _norm(s): s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower(); return re.sub(r'\s+',' ', s)
    move_candidates = [c for c in data.columns if _norm(c) in ['ak','move','rep','segment','cycle','trial','action']]
    acc_candidates  = [c for c in data.columns if any(k in _norm(c) for k in ['acc','accelerometer'])]
    gyr_candidates  = [c for c in data.columns if any(k in _norm(c) for k in ['gyr','gyro'])]
    print("\n[SUMMARY]"); print(f" - 총 CSV 파일 수: {total_files}개"); print(f" - 총 로우 수: {len(data):,}")
    print(f" - 라벨 분포:\n{data['label'].value_counts(dropna=False).to_string()}")
    print(f" - move 후보: {move_candidates if move_candidates else '[NONE]'}")
    print(f" - acc 후보: {acc_candidates[:6]}")
    print(f" - gyr 후보: {gyr_candidates[:6]}")
    return data, label_map_used

imu_df, label_map_used = load_labeled_imu(DATASET_ROOT)

# =============================================
# MOVE‑level preprocessing (Event‑anchored time‑normalize)
# =============================================

# -------- Config --------
FS = 50.0
L  = 128
MIN_MOVE_SEC = 0.50
GRAVITY_ALIGN = True
LPF_CUTOFF_HZ = 1.5

# Embedded‑friendly time strategy:
#   'stretch' : resample to EXACTLY L with event anchored at EVENT_ANCHOR (mask=ones)
#   'padcrop' : original behavior (pad/crop + mask)
TIME_MODE    = 'stretch'        # ⬅ 기본값: 임베디드 호환
EVENT_ANCHOR = 0.50             # event placed at L*anchor
PAD_ALIGN    = 'event'          # used if TIME_MODE='padcrop'
PAD_VALUE    = 'zero'

NPZ_PATH   = EXTRACT_DIR / f'move_pad_fs{int(FS)}_L{L}_{TIME_MODE}.npz'
SCALE_JSON = EXTRACT_DIR / f'move_pad_fs{int(FS)}_L{L}_scaler.json'
DEPLOY_JSON = EXTRACT_DIR / f'deploy_config.json'
REP_NPZ    = EXTRACT_DIR / f'rep_calib.npz'

# -------- Column mapping / resample helpers --------

def _normalize_name(s):
    s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower(); return re.sub(r'\s+',' ', s)

TIME_CAND = ['time','timestamp','time_ms','t','timestep','ts']
MOVE_CAND = ['ak','move','rep','segment','cycle','trial','action']

CANONICAL = {
    'ax':['ax','acc_x','accx','accelerometer x','a_x','acc-x','accx','accX','AccX'],
    'ay':['ay','acc_y','accy','accelerometer y','a_y','acc-y','accy','accY','AccY'],
    'az':['az','acc_z','accz','accelerometer z','a_z','acc-z','accz','accZ','AccZ'],
    'gx':['gx','gyro_x','gyrox','gyro x','g_x','gyrx','gyrX','GyroX'],
    'gy':['gy','gyro_y','gyroy','gyro y','g_y','gyry','gyrY','GyroY'],
    'gz':['gz','gyro_z','gyroz','gyro z','g_z','gyrz','gyrZ','GyroZ']
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_norm = {c:_normalize_name(c) for c in df.columns}
    # time
    time_col = None
    for c,n in cols_norm.items():
        if any(n == _normalize_name(k) for k in TIME_CAND) or 'time' in n:
            time_col = c; break
    # move
    move_col = None
    for c,n in cols_norm.items():
        if any(n == _normalize_name(k) for k in MOVE_CAND) or any(_normalize_name(k) in n for k in MOVE_CAND):
            move_col = c; break
    # 6 channels
    taken, mapping = set(), {}
    def find(keys):
        for c,n in cols_norm.items():
            if c in taken: continue
            if any(n == _normalize_name(k) for k in keys) or any(_normalize_name(k) in n for k in keys):
                taken.add(c); return c
        return None
    for canon, keys in CANONICAL.items():
        hit = find(keys)
        if hit is None:
            raise ValueError(f"[ERROR] 6채널 매핑 실패: {canon} 누락. 현재 컬럼 예: {list(df.columns)[:12]}")
        mapping[canon] = hit
    colmap = {v:k for k,v in mapping.items()}
    if time_col is not None: colmap[time_col] = 'time'
    if move_col is not None: colmap[move_col] = 'move'
    df2 = df.rename(columns=colmap)
    keep = (['time'] if 'time' in df2.columns else []) + ['ax','ay','az','gx','gy','gz','label','source_file','source_dir']
    if 'move' in df2.columns: keep.append('move')
    keep = [c for c in keep if c in df2.columns]
    return df2[keep]


# --- base resample to uniform time (keeps native length) ---

def resample_uniform(df_seq: pd.DataFrame, fs=FS):
    if 'time' in df_seq.columns:
        t = pd.to_numeric(df_seq['time'], errors='coerce').to_numpy()
        if len(t)==0 or np.all(np.isnan(t)): return None, 0
        t = t - np.nanmin(t)
        span = np.nanmax(t) - np.nanmin(t)
        if span > 200: t = t/1000.0  # ms→s
        order = np.argsort(t); t = t[order]
        X = df_seq[['ax','ay','az','gx','gy','gz']].to_numpy(dtype=float)[order]
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        dur = float(t[-1]) if len(t)>1 else (len(df_seq)/fs)
        n_target = max(2, int(round(dur*fs)))
        t_new = np.linspace(0.0, dur, n_target)
        Xr = np.vstack([np.interp(t_new, t, X[:,i]) for i in range(X.shape[1])]).T
        return Xr, dur
    X = df_seq[['ax','ay','az','gx','gy','gz']].to_numpy(dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    dur = len(X)/fs
    return X.copy(), dur


# --- stretch to EXACT L with event anchored at L*anchor ---

def stretch_event_anchor(X, event_idx, L, anchor=0.5):
    n, C = X.shape
    left_len  = int(round(anchor * L))
    right_len = L - left_len
    # left: [0 .. e_idx]  (include event)
    n1 = int(event_idx) + 1
    # right: (e_idx .. end] but exclude event itself
    n2 = max(0, n - n1)
    out = np.zeros((L, C), dtype=np.float32)
    pos = 0
    # left part
    if left_len > 0 and n1 > 0:
        x_idx = np.arange(n1, dtype=np.float32)
        t_new = np.linspace(0, n1-1, left_len, dtype=np.float32)
        for c in range(C): out[pos:pos+left_len, c] = np.interp(t_new, x_idx, X[:n1, c])
    pos += left_len
    # right part
    if right_len > 0 and n2 > 0:
        x_idx = np.arange(n2, dtype=np.float32)
        t_new = np.linspace(0, n2-1, right_len, dtype=np.float32)
        for c in range(C): out[pos:pos+right_len, c] = np.interp(t_new, x_idx, X[n1:, c])
    # mask = ones (no padding)
    m = np.ones(L, dtype=np.float32)
    return out, m


# -------- Gravity align (scipy optional) --------
try:
    from scipy.signal import butter, filtfilt
    def _lpf(x, fs, fc=LPF_CUTOFF_HZ):
        b,a = butter(2, fc/(fs/2), btype='low'); return filtfilt(b,a,x,axis=0)
except Exception:
    def _lpf(x, fs, fc=LPF_CUTOFF_HZ):
        k=max(1,int(round(fs/max(fc,1e-3)))); w=np.ones(k)/k
        return np.vstack([np.convolve(x[:,i], w, mode='same') for i in range(x.shape[1])]).T


def _rodrigues_R(a,b,eps=1e-8):
    a=a/np.linalg.norm(a); b=b/np.linalg.norm(b)
    v=np.cross(a,b); s=np.linalg.norm(v); c=float(np.dot(a,b))
    if s<eps:
        if c>0: return np.eye(3)
        axis = np.array([1,0,0]) if abs(a[0])<0.9 else np.array([0,1,0])
        v=np.cross(a,axis); v/=np.linalg.norm(v)
        K=np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
        return np.eye(3)+2*K@K
    K=np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[ -v[1],v[0],0]])
    return np.eye(3)+K+K@K*((1-c)/(s**2))


def gravity_align_and_linearize(X_acc, X_gyr, fs=FS):
    g = _lpf(X_acc, fs); g_mean = g.mean(axis=0)
    R = np.eye(3) if np.linalg.norm(g_mean)<1e-9 else _rodrigues_R(g_mean, np.array([0,0,1.0]))
    acc_w = (R @ X_acc.T).T; gyr_w = (R @ X_gyr.T).T
    acc_w[:,2] -= acc_w[:,2].mean()  # remove DC bias
    return acc_w, gyr_w


# -------- Move segmentation --------

def find_move_column(df, hints=('AK','move','Move','MOVE','rep','segment','cycle','trial','action')):
    cols=list(df.columns)
    for h in hints:
        if h in cols: return h
    for c in cols:
        n=_normalize_name(c)
        if any(k in n for k in ['move','segment','rep','cycle','trial','action','ak']): return c
    raise ValueError('move 열을 찾지 못함')


def contiguous_runs(mask_bool):
    m=np.asarray(mask_bool, dtype=bool)
    if m.size==0: return []
    diff=np.diff(np.concatenate(([False],m,[False])).astype(int))
    st=np.where(diff==1)[0]; en=np.where(diff==-1)[0]
    return list(zip(st,en))


def split_by_move(df, move_col):
    v=df[move_col]
    if v.dtype=='O' or str(v.dtype).startswith('category'):
        val=v.astype(str).str.lower().fillna('')
        return contiguous_runs(val.str.contains('move').to_numpy())
    if v.dropna().isin([0,1,True,False]).all():
        return contiguous_runs(v.astype(bool).fillna(False).to_numpy())
    arr=v.to_numpy()
    if np.isnan(arr).any(): arr=pd.Series(arr).fillna(method='ffill').fillna(method='bfill').to_numpy()
    ch=np.r_[True, arr[1:]!=arr[:-1], True]; idx=np.flatnonzero(ch)
    return [(idx[i], idx[i+1]) for i in range(len(idx)-1)]


def detect_bottom_index(acc_z):
    if len(acc_z)<4: return len(acc_z)//2
    return int(np.argmin(acc_z))


# -------- Pad/Crop (legacy) --------

def place_with_mask(X, L, align='event', pad_value='zero', event_idx=None, anchor=0.5):
    n, C = X.shape
    if n > L:
        if align == 'right': start = n - L
        elif align == 'center': start = max(0, (n - L)//2)
        elif align == 'event' and event_idx is not None:
            target = int(round(anchor * L)); start = int(np.clip(event_idx - target, 0, n - L))
        else: start = 0
        Xsub = X[start:start+L]; mask = np.ones(L, dtype=np.float32); return Xsub, mask
    if pad_value == 'zero': canvas = np.zeros((L, C), dtype=float)
    elif pad_value == 'mean': canvas = np.repeat(X.mean(axis=0, keepdims=True), L, axis=0)
    elif pad_value == 'edge': canvas = np.repeat(X[-1:, :], L, axis=0)
    else: canvas = np.zeros((L, C), dtype=float)
    if align == 'right': start = L - n
    elif align == 'center': start = (L - n)//2
    elif align == 'event' and event_idx is not None:
        target = int(round(anchor * L)); start = int(np.clip(target - event_idx, 0, L - n))
    else: start = 0
    canvas[start:start+n] = X
    mask = np.zeros(L, dtype=np.float32); mask[start:start+n] = 1.0
    return canvas, mask


# -------- Build by move --------
from sklearn.model_selection import StratifiedShuffleSplit

def build_by_move(imu_df):
    df_std = standardize_columns(imu_df.copy())
    has_src = all(c in df_std.columns for c in ['source_dir','source_file'])
    groups = df_std.groupby(['source_dir','source_file'], sort=False) if has_src else None
    try: mv_global = find_move_column(imu_df)
    except Exception: mv_global=None

    Xs, Ms, ys, durs, srcs = [], [], [], [], []
    it = (((d,f), g) for (d,f), g in groups) if groups is not None else [(('all','all'), df_std)]

    dropped_short = 0
    for key, g in it:
        (dname, fname) = key if isinstance(key, tuple) else ('all','all')
        g = g.reset_index(drop=True)
        mv_col = mv_global if (mv_global and mv_global in g.columns) else None
        if mv_col is None:
            try: mv_col = find_move_column(g)
            except Exception: print(f"[WARN] move 열 없음: {dname}/{fname}"); continue
        segs = split_by_move(g, mv_col)
        if not segs:
            print(f"[WARN] move 구간 없음: {dname}/{fname}"); continue
        for s,e in segs:
            seg = g.iloc[s:e]
            if len(seg) < int(MIN_MOVE_SEC*FS): dropped_short += 1; continue
            Xr, dur = resample_uniform(seg, FS)
            if Xr is None or len(Xr)==0: continue
            if GRAVITY_ALIGN:
                acc_w, gyr_w = gravity_align_and_linearize(Xr[:,:3], Xr[:,3:], FS)
                Xr = np.concatenate([acc_w, gyr_w], axis=1)
            event_idx = detect_bottom_index(Xr[:,2])
            if TIME_MODE=='stretch':
                Xi, Mi = stretch_event_anchor(Xr, event_idx, L, anchor=EVENT_ANCHOR)
            else:
                Xi, Mi = place_with_mask(Xr, L, align=PAD_ALIGN, pad_value=PAD_VALUE, event_idx=event_idx, anchor=EVENT_ANCHOR)
            Xi = np.nan_to_num(Xi, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            y = int(seg['label'].mode().iloc[0]) if 'label' in seg.columns else 0
            Xs.append(Xi); Ms.append(Mi.astype(np.float32)); ys.append(y); durs.append(dur)
            srcs.append(f"{dname}/{fname}#move[{s}:{e}]")
    if len(Xs)==0: raise RuntimeError('생성된 move가 없습니다. (move/AK 열을 확인하세요)')
    X = np.stack(Xs, axis=0); M = np.stack(Ms, axis=0); y = np.asarray(ys, dtype=np.int32); durs = np.asarray(durs, dtype=np.float32)
    print("\n[MOVE SUMMARY]"); print(' - N:', len(X)); print(' - X:', X.shape, '(N,L,6)'); print(' - M:', M.shape, '(N,L)'); print(' - y dist:\n', pd.Series(y).value_counts(dropna=False).to_string())
    print(' - dropped(short):', dropped_short)
    return X, M, y, durs, srcs


# -------- Split & Masked Z‑score --------

def safe_split_idx(y, seed=42, test_size=0.2, val_size=0.5):
    uniq, cnt = np.unique(y, return_counts=True)
    if len(uniq)<2: raise RuntimeError('최소 2개 클래스 필요')
    if (cnt<2).any() or len(y)<5:
        print('[WARN] 적은 샘플 → 비계층 분할')
        idx=np.arange(len(y)); rng=np.random.RandomState(seed); rng.shuffle(idx)
        n_test=max(1,int(round(test_size*len(y)))); te=idx[:n_test]; rem=idx[n_test:]
        n_val=max(1,int(round(val_size*len(rem)))); va=rem[:n_val]; tr=rem[n_val:]
        return tr, va, te
    sss1=StratifiedShuffleSplit(n_splits=1,test_size=test_size,random_state=seed)
    tr, te = next(sss1.split(np.zeros(len(y)), y))
    y_tmp = y[te]
    sss2=StratifiedShuffleSplit(n_splits=1,test_size=val_size,random_state=seed)
    va_rel, te_rel = next(sss2.split(np.zeros(len(y_tmp)), y_tmp))
    va, te = te[va_rel], te[te_rel]
    return tr, va, te


def fit_masked_zscore(X, M):
    w = M[..., None]
    count = w.sum(axis=(0,1)); count[count==0]=1.0
    mu = (X*w).sum(axis=(0,1))/count
    var= ((X - mu)**2 * w).sum(axis=(0,1))/count
    sigma = np.sqrt(np.maximum(var, 1e-12))
    return mu.astype(np.float32), sigma.astype(np.float32)


def apply_zscore_masked(X, mu, sigma):
    return ((X - mu) / sigma).astype(np.float32)


# -------- Run & Save --------
X, M, y, durs, srcs = build_by_move(imu_df)
tr, va, te = safe_split_idx(y, seed=42, test_size=0.2, val_size=0.5)
Xtr, ytr, Mtr = X[tr], y[tr], M[tr]; Xva, yva, Mva = X[va], y[va], M[va]; Xte, yte, Mte = X[te], y[te], M[te]

mu, sigma = fit_masked_zscore(Xtr, Mtr)
Xtr_n = apply_zscore_masked(Xtr, mu, sigma)
Xva_n = apply_zscore_masked(Xva, mu, sigma)
Xte_n = apply_zscore_masked(Xte, mu, sigma)

np.savez_compressed(
    NPZ_PATH,
    X_train=Xtr_n, y_train=ytr, mask_train=Mtr,
    X_val=Xva_n,   y_val=yva,   mask_val=Mva,
    X_test=Xte_n,  y_test=yte,  mask_test=Mte
)

with open(SCALE_JSON, 'w') as f:
    json.dump({'mu':mu.tolist(),'sigma':sigma.tolist(),
               'fs':FS,'L':L,
               'time_mode':TIME_MODE,
               'event_anchor':EVENT_ANCHOR,
               'pad_align':PAD_ALIGN,'pad_value':PAD_VALUE,
               'gravity_align':GRAVITY_ALIGN,'lpf_cutoff_hz':LPF_CUTOFF_HZ,
               'channels':['ax','ay','az','gx','gy','gz']}, f, indent=2)

# minimal deploy config (MCU 참고용)
deploy_cfg = {
    'fs': FS,
    'L': L,
    'gravity_align': GRAVITY_ALIGN,
    'lpf_cutoff_hz': LPF_CUTOFF_HZ,
    'time_mode': TIME_MODE,
    'event_anchor': EVENT_ANCHOR,
    'channels': ['ax','ay','az','gx','gy','gz'],
    'norm_clip': 3.0  # (선택) MCU에서 입력을 [-3,3]로 클리핑 후 사용
}
with open(DEPLOY_JSON, 'w') as f: json.dump(deploy_cfg, f, indent=2)

# representative subset for quantization / sanity testing
rng = np.random.default_rng(42)
N = Xtr_n.shape[0] + Xva_n.shape[0] + Xte_n.shape[0]
allX = np.concatenate([Xtr_n, Xva_n, Xte_n], axis=0)
sel = rng.choice(allX.shape[0], size=int(min(256, allX.shape[0])), replace=False)
np.savez_compressed(REP_NPZ, rep=allX[sel].astype(np.float32))

print(f"\n[OK] saved → {NPZ_PATH}")
print(f"[OK] scaler → {SCALE_JSON}")
print(f"[OK] deploy_config → {DEPLOY_JSON}")
print(f"[OK] rep_calib → {REP_NPZ}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] 사용할 ZIP: IMU_all.zip
[OK] 압축 해제 완료 → /content/IMU_DATA_extracted
[INFO] DATASET_ROOT(auto): /content/IMU_DATA_extracted
[INFO] 라벨 매핑: {'label_0': 0, 'label_1': 1}
[OK] label_0 → 1개 파일
    • IMU_Label_Plus_ALL.csv
[OK] label_1 → 1개 파일
    • IMU_label_1_jungro.csv

[SUMMARY]
 - 총 CSV 파일 수: 2개
 - 총 로우 수: 45,143
 - 라벨 분포:
label
0    36535
1     8608
 - move 후보: ['move']
 - acc 후보: []
 - gyr 후보: []

[MOVE SUMMARY]
 - N: 771
 - X: (771, 128, 6) (N,L,6)
 - M: (771, 128) (N,L)
 - y dist:
 0    621
1    150
 - dropped(short): 0

[OK] saved → /content/IMU_DATA_extracted/move_pad_fs50_L128_stretch.npz
[OK] scaler → /content/IMU_DATA_extracted/move_pad_fs50_L128_scaler.json
[OK] deploy_config → /content/IMU_DATA_extracted/deploy_config.json
[OK] rep_calib → /content/IMU_DATA_extracted/rep_calib.npz


In [ ]:
# =============================================
# TCN Training (STM32Cube.AI–compatible, single-input)
# - No Lambda/custom ops, single Input (L,C)
# - Layers: Conv1D + BatchNorm + ReLU + Dropout + Add + GAP1D + Dense
# - Optional dilations (set USE_DILATION=False if Cube.AI balks)
# - Mixed precision for training OK; export as FP32 .h5 for Cube.AI
# - Uses NPZ from the preprocessing step (move_pad_fs* .npz)
# =============================================

import os, json, math, random, glob
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# -------------------- Repro / GPU mem --------------------
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception as e:
    print('[WARN] set_memory_growth skipped:', e)

# Mixed precision for speed/mem; export will cast to FP32
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# -------------------- Paths / NPZ discovery --------------------
EXTRACT_DIR = Path('/content/IMU_DATA_extracted')
NPZ_CAND = None
cands = sorted(EXTRACT_DIR.glob('move_pad_fs*_L*_*.npz'), key=lambda p: p.stat().st_mtime, reverse=True)
if cands:
    NPZ_CAND = cands[0]
else:
    raise FileNotFoundError('NPZ not found under /content/IMU_DATA_extracted — run preprocessing first.')
print('[INFO] NPZ:', NPZ_CAND)
OUT_ROOT = Path('/content/tcn_runs_cubeai'); OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------- Train config --------------------
L = int(next(s for s in NPZ_CAND.stem.split('_') if s.startswith('L')).lstrip('L'))
BATCH = 32
EPOCHS = 50
INIT_LR = 2.0e-3
PATIENCE = 7
MIN_VALID_RATIO = 0.60

# Model size (embedded friendly)
WIDTH = 48           # 32~64
KERNEL_SIZE = 5
USE_DILATION = True
DILATIONS = [1,2,4,8,16] if USE_DILATION else [1,1,1,1,1]
DROPOUT_P = 0.20
HEAD_UNITS = 96

# -------------------- IO utils --------------------

def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=True)
    Xtr, ytr, Mtr = d['X_train'], d['y_train'], d['mask_train']
    Xva, yva, Mva = d['X_val'],   d['y_val'],   d['mask_val']
    Xte, yte, Mte = d['X_test'],  d['y_test'],  d['mask_test']
    if Mtr.ndim == 2: Mtr = Mtr[...,None]
    if Mva.ndim == 2: Mva = Mva[...,None]
    if Mte.ndim == 2: Mte = Mte[...,None]
    return (Xtr, ytr, Mtr), (Xva, yva, Mva), (Xte, yte, Mte)


def filter_by_valid_ratio(X, y, M, thr=MIN_VALID_RATIO):
    vr = M.mean(axis=1).reshape(-1)
    keep = vr >= thr
    return X[keep], y[keep], M[keep]


def make_class_weights(y):
    classes = np.unique(y)
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, cw)}

# -------------------- Augmentation (train only) --------------------
AUG = dict(yaw_deg=15.0, max_shift=4, noise_std=0.02, drop_p=0.10)

def augment_np(x, m):
    # x:(L,C), m:(L,1)
    L_, C_ = x.shape
    if C_ >= 6:
        th = np.deg2rad(np.random.uniform(-AUG['yaw_deg'], AUG['yaw_deg']))
        c, s = np.cos(th), np.sin(th)
        ax, ay = x[:,0].copy(), x[:,1].copy()
        gx, gy = x[:,3].copy(), x[:,4].copy()
        x[:,0] = c*ax - s*ay
        x[:,1] = s*ax + c*ay
        x[:,3] = c*gx - s*gy
        x[:,4] = s*gx + c*gy
    sh = np.random.randint(-AUG['max_shift'], AUG['max_shift']+1)
    if sh != 0:
        x = np.roll(x, sh, axis=0)
        m = np.roll(m, sh, axis=0)
    x = x + np.random.normal(0.0, AUG['noise_std'], size=x.shape).astype(np.float32)
    if np.random.rand() < AUG['drop_p']:
        t0 = np.random.randint(0, max(1, L_-8))
        t1 = min(L_, t0 + np.random.randint(4, 12))
        x[t0:t1, :] *= 0.0
        m[t0:t1, :] *= 0.0
    return x.astype(np.float32), m.astype(np.float32)


def tf_augment(x, m, y):
    x, m = tf.numpy_function(lambda a,b: augment_np(a,b), [x, m], [tf.float32, tf.float32])
    x.set_shape([L, None])
    m.set_shape([L, 1])
    return x, m, y

# -------------------- Masked → Single-input --------------------
# x <- x * m * (L / valid)  (so GAP equals masked average)

def to_single_input(x, m, y):
    x = tf.cast(x, tf.float32)
    m = tf.cast(m, tf.float32)
    valid = tf.reduce_sum(m)
    scale = tf.cast(L, tf.float32) / (valid + 1e-6)
    x_masked = x * m * scale
    return x_masked, y

# -------------------- Datasets --------------------

def make_ds(X, M, y, training=False):
    X = X.astype('float32'); M = M.astype('float32'); y = y.astype('int32')
    C = X.shape[-1]
    ds = tf.data.Dataset.from_tensor_slices((X, M, y))
    if training:
        buf = int(min(len(y), 4096))
        ds = ds.shuffle(buffer_size=buf, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(tf_augment, num_parallel_calls=1, deterministic=True)
    ds = ds.map(to_single_input, num_parallel_calls=1, deterministic=True)
    ds = ds.map(lambda x, y: (tf.ensure_shape(x, [L, C]), y), num_parallel_calls=1)
    ds = ds.batch(BATCH, drop_remainder=False).prefetch(1)
    return ds

# -------------------- Model (Cube.AI–safe) --------------------

def tcn_block(x, filters, k=5, d=1, pdrop=0.2, name='tcn'):
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=True, name=f'{name}_conv1')(x)
    y = layers.BatchNormalization(name=f'{name}_bn1')(y)
    y = layers.Activation('relu', name=f'{name}_relu1')(y)
    y = layers.Dropout(pdrop, name=f'{name}_drop1')(y)
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=True, name=f'{name}_conv2')(y)
    y = layers.BatchNormalization(name=f'{name}_bn2')(y)
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding='same', use_bias=True, name=f'{name}_proj')(x)
    out = layers.Add(name=f'{name}_add')([x, y])
    out = layers.Activation('relu', name=f'{name}_relu2')(out)
    return out


def build_tcn_cubeai(L, C, num_classes, filters=48, k=5, dils=(1,2,4,8,16), pdrop=0.2):
    x_in = layers.Input(shape=(L, C), name='x')
    x = layers.Conv1D(filters, 3, padding='same', use_bias=True, name='stem_conv')(x_in)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.Activation('relu', name='stem_relu')(x)
    for i, d in enumerate(dils):
        x = tcn_block(x, filters=filters, k=k, d=d, pdrop=pdrop, name=f'tcn{i+1}')
    x = layers.GlobalAveragePooling1D(name='gap')(x)
    h = layers.Dense(HEAD_UNITS, activation='relu', name='head_fc')(x)
    h = layers.Dropout(0.30, name='head_drop')(h)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32', name='pred')(h)
    return keras.Model(x_in, out, name='tcn_cls_cubeai')

# -------------------- Callbacks / LR --------------------

def cosine_decay(epoch, lr):
    t = min(epoch, EPOCHS)
    return 0.5 * INIT_LR * (1.0 + math.cos(math.pi * t / EPOCHS))

class ValF1Callback(keras.callbacks.Callback):
    def __init__(self, ds_val, patience=7):
        super().__init__(); self.ds_val=ds_val; self.best=-1.0; self.patience=patience; self.wait=0
    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.ds_val:
            pb = self.model.predict(xb, verbose=0)
            y_true.append(yb.numpy()); y_pred.append(pb.argmax(axis=1))
        y_true = np.concatenate(y_true, 0); y_pred = np.concatenate(y_pred, 0)
        f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n[VAL] macro-F1={f1m:.4f}")
        if f1m > self.best + 1e-6:
            self.best = f1m; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"[EarlyStopping] No improvement in {self.patience} epochs (val F1)")
                self.model.stop_training = True

# -------------------- Train --------------------

def train_and_export(npz_path: Path):
    (Xtr, ytr, Mtr), (Xva, yva, Mva), (Xte, yte, Mte) = load_npz(npz_path)

    # remove overly padded segments
    Xtr, ytr, Mtr = filter_by_valid_ratio(Xtr, ytr, Mtr)
    Xva, yva, Mva = filter_by_valid_ratio(Xva, yva, Mva)
    Xte, yte, Mte = filter_by_valid_ratio(Xte, yte, Mte)

    C = Xtr.shape[-1]
    n_classes = int(max(ytr.max(), yva.max(), yte.max()) + 1)
    assert Xtr.shape[1] == L
    print('[INFO] Shapes: tr', Xtr.shape, 'va', Xva.shape, 'te', Xte.shape, 'classes', n_classes)

    ds_tr = make_ds(Xtr, Mtr, ytr, training=True)
    ds_va = make_ds(Xva, Mva, yva, training=False)
    ds_te = make_ds(Xte, Mte, yte, training=False)

    model = build_tcn_cubeai(L, C, n_classes, filters=WIDTH, k=KERNEL_SIZE, dils=DILATIONS, pdrop=DROPOUT_P)
    opt = keras.optimizers.Adam(learning_rate=INIT_LR)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    run_dir = OUT_ROOT / f'L{L}_C{C}_W{WIDTH}_K{KERNEL_SIZE}_dil{int(USE_DILATION)}'
    run_dir.mkdir(parents=True, exist_ok=True)

    cbs = [
        ValF1Callback(ds_va, patience=PATIENCE),
        keras.callbacks.ModelCheckpoint(str(run_dir / 'best.keras'), monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=max(2, PATIENCE//3), min_lr=1e-5, verbose=1),
        keras.callbacks.LearningRateScheduler(cosine_decay, verbose=0),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE, restore_best_weights=True, mode='max', verbose=1),
    ]

    cw = make_class_weights(ytr)

    hist = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS, class_weight=cw, verbose=1, callbacks=cbs)

    # quick eval
    va_eval = model.evaluate(ds_va, verbose=0)
    te_eval = model.evaluate(ds_te, verbose=0)
    print(f"[VAL] loss={va_eval[0]:.4f}, acc={va_eval[1]:.4f}")
    print(f"[TEST] loss={te_eval[0]:.4f}, acc={te_eval[1]:.4f}")

    # -------------------- Export: FP32 HDF5 --------------------
    # Reset policy to float32 then clone & cast weights for robust Cube.AI import
    mixed_precision.set_global_policy('float32')
    model_f32 = keras.models.clone_model(model)
    model_f32.build(model.input_shape)
    model_f32.set_weights([w.astype(np.float32) for w in model.get_weights()])

    # sanity forward pass
    xb = next(iter(ds_te.take(1)))[0]
    _ = model_f32.predict(xb, verbose=0)

    h5_path = run_dir / 'final_export_cubeai.h5'
    model_path = run_dir / 'final.keras'
    model_f32.save(h5_path, include_optimizer=False)
    model.save(model_path, include_optimizer=False)
    print('[SAVED] H5 for Cube.AI →', h5_path)
    print('[SAVED] Keras backup →', model_path)

    # Save a minimal signature for debugging
    with open(run_dir / 'export_meta.json', 'w') as f:
        json.dump({
            'npz': str(npz_path), 'L': L, 'C': int(C), 'classes': n_classes,
            'width': WIDTH, 'kernel': KERNEL_SIZE, 'dilations': DILATIONS,
            'use_dilation': USE_DILATION
        }, f, indent=2)

    # Reports
    def predict_ds(ds):
        probs, ytrue = [], []
        for xb, yb in ds:
            probs.append(model_f32.predict(xb, verbose=0))
            ytrue.append(yb.numpy())
        return np.concatenate(probs, 0), np.concatenate(ytrue, 0)

    pv, yv = predict_ds(ds_va)
    pt, yt = predict_ds(ds_te)
    yv_pred, yt_pred = pv.argmax(1), pt.argmax(1)

    def save_report(y_true, y_pred, path):
        rep = classification_report(y_true, y_pred, digits=4, zero_division=0)
        cm  = confusion_matrix(y_true, y_pred)
        with open(path, 'w', encoding='utf-8') as f:
            f.write(rep + '\n')
            f.write('CM (rows=true, cols=pred):\n')
            f.write(np.array2string(cm, separator=' ') + '\n')
        print('\n[REPORT]\n' + rep)
        print('CM:\n', cm)

    save_report(yv, yv_pred, run_dir / 'val_report.txt')
    save_report(yt, yt_pred, run_dir / 'test_report.txt')

    return str(h5_path)

# -------------------- Main --------------------
if __name__ == '__main__':
    export_h5 = train_and_export(NPZ_CAND)
    print('\n[OK] Exported:', export_h5)


[INFO] NPZ: /content/IMU_DATA_extracted/move_pad_fs50_L128_stretch.npz
[INFO] Shapes: tr (616, 128, 6) va (77, 128, 6) te (78, 128, 6) classes 2


Model: "tcn_cls_cubeai"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 6)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv1D)  │ (None, 128, 48)   │        912 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 128, 48)   │        192 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu           │ (None, 128, 48)   │          0 │ stem_bn[0][0]     │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │     11,568 │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_bn1            │ (None, 128, 48)   │        192 │ tcn1_conv1[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu1          │ (None, 128, 48)   │          0 │ tcn1_bn1[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 48)   │          0 │ tcn1_relu1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_bn2            │ (None, 128, 48)   │        192 │ tcn1_conv2[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ stem_relu[0][0],  │
│                     │                   │            │ tcn1_bn2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu2          │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn1_relu2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_bn1            │ (None, 128, 48)   │        192 │ tcn2_conv1[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu1          │ (None, 128, 48)   │          0 │ tcn2_bn1[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 48)   │          0 │ tcn2_relu1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_bn2            │ (None, 128, 48)   │        192 │ tcn2_conv2[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_relu2[0][0]

 Total params: 123,602 (482.82 KB)

 Trainable params: 122,546 (478.70 KB)

 Non-trainable params: 1,056 (4.12 KB)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 0.6633 - loss: 0.7079


[VAL] macro-F1=0.1630

Epoch 1: val_accuracy improved from -inf to 0.19481, saving model to /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.6668 - loss: 0.7017 - val_accuracy: 0.1948 - val_loss: 3.4275 - learning_rate: 0.0020
Epoch 2/50
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8015 - loss: 0.4623
[VAL] macro-F1=0.2311

Epoch 2: val_accuracy improved from 0.19481 to 0.24675, saving model to /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.8017 - loss: 0.4618 - val_accuracy: 0.2468 - val_loss: 1.6184 - learning_rate: 0.0020
Epoch 3/50
18/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8494 - loss: 0.2949
[VAL] macro-F1=0.2929

Epoch 3: val_accuracy improved from 0.24675 to 0.29870, saving model to /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.8528 - loss: 0.2928 - val_accuracy: 0.2987 - v

[SAVED] H5 for Cube.AI → /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/final_export_cubeai.h5
[SAVED] Keras backup → /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/final.keras

[REPORT]
              precision    recall  f1-score   support

           0     1.0000    0.9839    0.9919        62
           1     0.9375    1.0000    0.9677        15

    accuracy                         0.9870        77
   macro avg     0.9688    0.9919    0.9798        77
weighted avg     0.9878    0.9870    0.9872        77

CM:
 [[61  1]
 [ 0 15]]

[REPORT]
              precision    recall  f1-score   support

           0     0.9836    0.9524    0.9677        63
           1     0.8235    0.9333    0.8750        15

    accuracy                         0.9487        78
   macro avg     0.9036    0.9429    0.9214        78
weighted avg     0.9528    0.9487    0.9499        78

CM:
 [[60  3]
 [ 1 14]]

[OK] Exported: /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/final_export_cubeai.h5


In [ ]:
# ===================== Robust FP32 Export (No global `model` required) =====================
# - Finds latest saved Keras model under /content/tcn_runs_cubeai
# - Rebuilds an FP32-only graph (no dtype mixing) with same TCN topology
# - Copies weights (cast to float32) and saves export_fp32_no_cast.h5
# - Falls back to clone_model path if named rebuild fails
import re, json, numpy as np, tensorflow as tf
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import mixed_precision

# 0) Locate the most recent saved Keras model from your training run
ROOT = Path('/content/tcn_runs_cubeai')
cands = sorted(
    [*ROOT.rglob('final.keras'), *ROOT.rglob('best.keras'), *ROOT.rglob('*.keras')],
    key=lambda p: p.stat().st_mtime, reverse=True
)
assert cands, "저장된 .keras 모델을 찾지 못했습니다. 학습 셀이 끝난 후 생성된 폴더를 확인하세요."
MODEL_SRC = cands[0]
print("[LOAD]", MODEL_SRC)

# 1) Load original model (compile off)
orig = keras.models.load_model(MODEL_SRC, compile=False)

# 2) Force global FP32 policy for clean export
mixed_precision.set_global_policy('float32')
tf.keras.backend.set_floatx('float32')

# 3) Infer shapes and TCN hyperparams from layer names
L = int(orig.input_shape[1]); C = int(orig.input_shape[2])
NUM_CLASSES = int(orig.output_shape[-1])

# parse dilations/width/kernel from layers named like tcn1_conv1, tcn2_conv1, ...
tcn_conv1_layers = []
for lyr in orig.layers:
    m = re.match(r"tcn(\d+)_conv1$", lyr.name)
    if m:
        tcn_conv1_layers.append((int(m.group(1)), lyr))
tcn_conv1_layers.sort(key=lambda x: x[0])
assert tcn_conv1_layers, "tcn*_conv1 레이어를 찾지 못했습니다. 학습 스크립트의 네이밍(tcn1_conv1 등)을 유지해주세요."

DILATIONS = [int(l.dilation_rate[0]) for _, l in tcn_conv1_layers]
KERNEL_SIZE = int(tcn_conv1_layers[0][1].kernel_size[0])
WIDTH = int(tcn_conv1_layers[0][1].filters)

# head units (default 96 if head_fc missing)
def _find(name):
    try: return orig.get_layer(name)
    except Exception: return None
head_fc = _find("head_fc")
HEAD_UNITS = int(head_fc.units) if head_fc is not None else 96

print(f"[EXPORT] L={L}, C={C}, classes={NUM_CLASSES}, width={WIDTH}, k={KERNEL_SIZE}, dils={DILATIONS}, head={HEAD_UNITS}")

# 4) Build clean FP32 topology (no explicit dtype on layers, dropout=0.0 for simplicity)
def tcn_block_export(x, filters, k=5, d=1, name='tcn'):
    y = keras.layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=True, name=f'{name}_conv1')(x)
    y = keras.layers.BatchNormalization(name=f'{name}_bn1')(y)
    y = keras.layers.Activation('relu', name=f'{name}_relu1')(y)
    y = keras.layers.Dropout(0.0, name=f'{name}_drop1')(y)
    y = keras.layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=True, name=f'{name}_conv2')(y)
    y = keras.layers.BatchNormalization(name=f'{name}_bn2')(y)
    skip = x
    if skip.shape[-1] != filters:
        skip = keras.layers.Conv1D(filters, 1, padding='same', use_bias=True, name=f'{name}_proj')(skip)
    out = keras.layers.Add(name=f'{name}_add')([skip, y])
    out = keras.layers.Activation('relu', name=f'{name}_relu2')(out)
    return out

def build_tcn_cubeai_export(L, C, num_classes, filters=48, k=5, dils=(1,2,4,8,16), head_units=96):
    x_in = keras.layers.Input(shape=(L, C), name='x')
    x = keras.layers.Conv1D(filters, 3, padding='same', use_bias=True, name='stem_conv')(x_in)
    x = keras.layers.BatchNormalization(name='stem_bn')(x)
    x = keras.layers.Activation('relu', name='stem_relu')(x)
    for i, d in enumerate(dils):
        x = tcn_block_export(x, filters=filters, k=k, d=d, name=f'tcn{i+1}')
    x = keras.layers.GlobalAveragePooling1D(name='gap')(x)
    x = keras.layers.Dense(head_units, activation='relu', name='head_fc')(x)
    out = keras.layers.Dense(num_classes, activation='softmax', name='pred')(x)  # no dtype forced
    return keras.Model(x_in, out, name='tcn_cls_cubeai_export')

try:
    export_model = build_tcn_cubeai_export(
        L, C, NUM_CLASSES, filters=WIDTH, k=KERNEL_SIZE, dils=tuple(DILATIONS), head_units=HEAD_UNITS
    )
    # Copy weights by matching layer names; cast to float32
    copied, skipped = 0, []
    for layer in export_model.layers:
        try:
            src = orig.get_layer(layer.name)
            w = src.get_weights()
            if w:
                w = [np.asarray(v, dtype=np.float32) for v in w]
                layer.set_weights(w)
                copied += 1
        except Exception:
            # non-weight layers or unmatched names are fine
            skipped.append(layer.name)
    print(f"[EXPORT] weights copied for {copied} layers")
except Exception as e:
    print("[WARN] Named rebuild failed, fallback to clone_model:", e)
    export_model = keras.models.clone_model(orig)
    export_model.build(orig.input_shape)
    export_model.set_weights([np.asarray(w, np.float32) for w in orig.get_weights()])

# 5) Save FP32 HDF5
OUT_DIR = Path("/content/tcn_export"); OUT_DIR.mkdir(parents=True, exist_ok=True)
H5_PATH = OUT_DIR / "export_fp32_no_cast.h5"
KERAS_PATH = OUT_DIR / "export_fp32_no_cast.keras"
export_model.save(H5_PATH, include_optimizer=False)
export_model.save(KERAS_PATH, include_optimizer=False)
print("[SAVED]", H5_PATH)
print("[SAVED]", KERAS_PATH)

# 6) Optional: quick numeric check if a sample is around
try:
    # try to load a npz to test a forward pass
    npz = sorted(Path('/content/IMU_DATA_extracted').glob('move_pad_fs*_L*_*.npz'))[-1]
    d = np.load(npz)
    Xte = d['X_test'].astype(np.float32)
    if Xte.shape[1] == L and Xte.shape[2] == C:
        _ = export_model.predict(Xte[:8], verbose=0)
        print("[CHECK] forward pass OK on sample batch")
except Exception as e:
    print("[CHECK] skipped:", e)

# 7) (Colab) Download to your PC automatically
try:
    from google.colab import files
    files.download(str(H5_PATH))
except Exception:
    pass


[LOAD] /content/tcn_runs_cubeai/L128_C6_W48_K5_dil1/final.keras


[EXPORT] L=128, C=6, classes=2, width=48, k=5, dils=[1, 2, 4, 8, 16], head=96
[EXPORT] weights copied for 24 layers
[SAVED] /content/tcn_export/export_fp32_no_cast.h5
[SAVED] /content/tcn_export/export_fp32_no_cast.keras
[CHECK] forward pass OK on sample batch


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>